# Measure average time and cost for cv parser processing one document

In [1]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisResult
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import AzureError
import os
import pprint

In [2]:
# AZURE_CONTENT_UNDERSTANDING_ENDPOINT - the endpoint to your Content Understanding resource.
endpoint = os.environ["CONTENTUNDERSTANDING_ENDPOINT"]
# CONTENT_UNDERSTANDING_KEY - your Content Understanding API key
key = os.environ["CONTENTUNDERSTANDING_KEY"]
credential = AzureKeyCredential(key)

In [3]:
# local file localtion
path_to_sample_document = "./examples/pdf/職務経歴書(サンプル).pdf"

In [33]:
analyer_candidates_list = [
    {
     # api version='2025-11-01' with gpt-5
     "api_version": "2025-11-01",
     "analyzer_id": "cv_parser_test_lk"},
    # api version='2026-06-01-preview' with gpt-5
    {"api_version": "2026-06-01-preview",
     "analyzer_id": "cv_parser_preview_api_2026_06_01"},
    # api version='2026-06-01-preview' with gpt-5-mini
    {
    "api_version": "2026-06-01-preview",
    "analyzer_id": "cv_parser_preview_api_2026_06_01_gpt5_mini"}
]

In [14]:
# According to Azure  Content Understanding in Foundry Tools pricing in https://azure.microsoft.com/en-us/pricing/details/content-understanding/#pricing
# and Azure OpenAI Service pricing in https://azure.microsoft.com/en-us/pricing/details/azure-openai/
price_dict = {
    # in our usecase, uploaded cv can be in pdf, docx, xlsx format and it require structural element detection from image-based documents.
    # We have to use standard meter here according to the instruction https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/pricing-explainer#example-1-document-processing-for-rag-workflows. 
    'documentPagesMinimal': 0.01 / 1000,
    'documentPagesStandard':  5 / 1000,
    'contextualizationTokens': 1 /1000000,
    'advancedContextualizationTokens': 3 /1000000,
    'tokens': {'text-embedding-3-large': 0.000158/1000, 
               'gpt-5-input': 1.25 / 1000000,
               'gpt-5-output': 10 /1000000,
               'gpt-5-mini-input': 0.25 / 1000000,
               'gpt-5-mini-output': 2 /1000000
}
}

In [7]:
def parse_cv(analyzer_id: str, path_to_sample_document:str, client: ContentUnderstandingClient):
    with open(path_to_sample_document, "rb") as f:
        poller = client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=f
        )
    result: AnalysisResult = poller.result()
    return result, poller

In [8]:
def caluate_total_cost(poller, price_dict):
    # Caluation is based on https://azure.microsoft.com/en-us/pricing/details/azure-openai/
    total_cost = 0
    for charge_item, item_count in poller.usage.items():
        if charge_item == "tokens":
            for sub_charge_item, sub_item_count in item_count.items():
                total_cost += price_dict[charge_item][sub_charge_item] * sub_item_count
            
        else:
            total_cost += price_dict[charge_item]*item_count
    print(f"Total cost (usd) per document(2-page pdf): {total_cost}")

## Mearue speed and cost for api_version=2025-11-01 with gpt-5

In [16]:
# API_VERSION - the API version to use.
api_version = analyer_candidates_list[0]["api_version"]

In [17]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = analyer_candidates_list[0]["analyzer_id"]

In [18]:
# Set up Content Understanding client.
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

In [19]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page pdf)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

22.5 s ± 1.05 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [21]:
# usage of tokens
poller.usage

{'documentPagesStandard': 2, 'contextualizationTokens': 2000, 'tokens': {'text-embedding-3-large': 1799, 'gpt-5-input': 8963, 'gpt-5-output': 1220}}

In [22]:
# Calulate total cost in usd for running cv parsing per document (2-page pdf)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.035687992


In [23]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社', 'spans': [{'offset': 273, 'length': 4}], 'confidence': 0.497, 'source': 'D(1,2.7439,3.9964,3.2805,4.0001,3.2805,4.1387,2.7439,4.1424)'}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'spans': [{'offset': 257, 'length': 7}], 'confidence': 0.548, 'source': 'D(1,0.9356,3.9983,1.6244,3.9998,1.6241,4.1403,0.9353,4.1389)'}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'spans': [{'offset': 265, 'length': 7}], 'confidence': 0.543, 'source': 'D(1,1.7755,3.9992,2.4878,4.0003,2.4876,4.1421,1.7753,4.1410)'}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'spans': [{'offset': 293, 'length': 11}], 'confidence': 0.711, 'source': 'D(1,1.5981,4.4793,3.0315,4.4889,3.0304,4.6523,1.5970,4.6427)'}, '業種': {'type': 'string', 'confidence': 0.863}, '職種': {'type': 'string', 'valueString': '経理課', 'sp

## Mearue speed and cost for api_version=2026-06-01-preview with gpt-5

In [8]:
# API_VERSION - the API version to use.
api_version = analyer_candidates_list[1]["api_version"]

In [9]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = analyer_candidates_list[1]["analyzer_id"]

In [10]:
# Set up Content Understanding client.
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

In [11]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page pdf)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

19.9 s ± 1.52 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [13]:
# usage of tokens
poller.usage

{'documentPagesStandard': 2, 'advancedContextualizationTokens': 2000, 'tokens': {'gpt-5-input': 11976, 'gpt-5-output': 1284}}

In [14]:
# Calulate total cost in usd for running cv parsing per document (2-page docx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.04381


In [15]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社', 'spans': [{'offset': 273, 'length': 4}], 'confidence': 0.552, 'source': 'D(1,2.7439,3.9964,3.2805,4.0001,3.2805,4.1387,2.7439,4.1424)'}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'spans': [{'offset': 257, 'length': 7}], 'confidence': 0.529, 'source': 'D(1,0.9356,3.9983,1.6244,3.9998,1.6241,4.1403,0.9353,4.1389)'}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'spans': [{'offset': 265, 'length': 7}], 'confidence': 0.533, 'source': 'D(1,1.7755,3.9992,2.4878,4.0003,2.4876,4.1421,1.7753,4.1410)'}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'spans': [{'offset': 293, 'length': 11}], 'confidence': 0.728, 'source': 'D(1,1.5981,4.4793,3.0315,4.4889,3.0304,4.6523,1.5970,4.6427)'}, '業種': {'type': 'string', 'confidence': 0.956}, '職種': {'type': 'string', 'valueString': '経理課', 'sp

### Under api_version=2026-06-01-preview with gpt-5

In [53]:
# API_VERSION - the API version to use.
api_version = analyer_candidates_list[1]["api_version"]

In [54]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = analyer_candidates_list[1]["analyzer_id"]

In [ ]:
# Set up Content Understanding client.
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

#### Measure time and cost it take to process document in word format

In [47]:
path_to_sample_document = "./examples/docx/職務経歴書(サンプル).docx"

In [48]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2 page docx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

11 s ± 1.58 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [50]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 1, 'advancedContextualizationTokens': 1000, 'tokens': {'gpt-5-input': 10290, 'gpt-5-output': 1134}}

In [51]:
# Calulate total cost in usd for running cv parsing per document (2-page docx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.0272125


In [52]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社▲▲', 'confidence': 1}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'confidence': 1}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'confidence': 1}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'confidence': 1}, '業種': {'type': 'string'}, '職種': {'type': 'string', 'valueString': '主任', 'confidence': 1}, '雇用形態': {'type': 'string', 'valueString': '正社員', 'confidence': 1}, '職務内容': {'type': 'string', 'valueString': '【日常業務】 ・仕訳、伝票処理（売掛金、買掛金） ・現金出納管理、預金口座管理 ・年末調整、償却資産管理 【決算関連業務】 ・月次・年次財務諸表作成（貸借対照表、損益計算書、キャッシュフロー計算書） ・原価計算配賦計算（原材料/人件費/経費） ・月次部門別損益集計', 'confidence': 1}, '実績と成果': {'type': 'string', 'valueString': '2012 年 4 月 同支店管理部 総務経理課 主任（部下 3 名） ※2014 年 8 月～2016 年 4 月まで産休・育休取得', 'confidence': 1}, '所属部門': {'type': 'string', 'valueString': '東京本社 管理部 経理課／大阪支店 管理部 総務経理課', 'confidence': 1

#### Measure time and cost it take to process document in excel format

In [56]:
path_to_sample_document = "./examples/xlsx/職務経歴書(サンプル).xlsx"

In [61]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page xlsx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

10.1 s ± 1.05 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [58]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 2, 'advancedContextualizationTokens': 2000, 'tokens': {'gpt-5-input': 12418, 'gpt-5-output': 747}}

In [59]:
# Calulate total cost in usd for running cv parsing per document (2-page xlsx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.029012500000000004


In [60]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社▲▲', 'confidence': 1}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'confidence': 1}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'confidence': 1}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'confidence': 1}, '業種': {'type': 'string'}, '職種': {'type': 'string', 'valueString': '主任', 'confidence': 1}, '雇用形態': {'type': 'string', 'valueString': '正社員', 'confidence': 1}, '職務内容': {'type': 'string', 'valueString': '【日常業務】 ・仕訳、伝票処理（売掛金、買掛金） ・現金出納管理、預金口座管理 ・年末調整、償却資産管理 【決算関連業務】 ・月次・年次財務諸表作成（貸借対照表、損益計算書、キャッシュフロー計算書） ・原価計算配賦計算（原材料/人件費/経費） ・月次部門別損益集計', 'confidence': 1}, '実績と成果': {'type': 'string'}, '所属部門': {'type': 'string', 'valueString': '東京本社 管理部 経理課／大阪支店 管理部 総務経理課', 'confidence': 1}}}, {'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社■■', 'confidence

### Under api_version=2025-11-01 with gpt-5

In [80]:
# API_VERSION - the API version to use.
api_version = analyer_candidates_list[0]["api_version"]

In [81]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = analyer_candidates_list[0]["analyzer_id"]

In [82]:
# Set up Content Understanding client.
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

#### Measure time and cost it take to process document in word format

In [68]:
path_to_sample_document = "./examples/docx/職務経歴書(サンプル).docx"

In [73]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2 page docx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

12.7 s ± 98.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [69]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [70]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 1, 'contextualizationTokens': 1000, 'tokens': {'text-embedding-3-large': 2232, 'gpt-5-input': 7231, 'gpt-5-output': 1060}}

In [71]:
# Calulate total cost in usd for running cv parsing per document (2-page docx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.021001406


In [72]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array'},
 'スキル': {'type': 'object', 'valueObject': {'技術': {'type': 'string', 'valueString': 'Word：見出し、目次、箇条書き、表組みを用いた資料作成が可能なレベル\n\nExcel ：数式、一般的な関数、および表組み・グラフ・ピボットテーブルを用いた資料作成が可能なレベル\n\nPowerPoint：マスター設定および図表を用いた資料作成が可能なレベル', 'confidence': 1}, '言語': {'type': 'string'}}},
 '自己PR': {'type': 'string', 'valueString': '＜業務における迅速性と正確性＞ 迅速性と正確性を心がけています。\n\n電子データ・紙媒体それぞれの情報を必要な時にすぐに取り出せるように適切に整理し、また効率的に作業を進められるよう Excel を活用してデータ管理をしていました。また就業開始時には必ずスケジュール進捗とタスクの優先度整理を行い、業務に臨みました。\n\n＜業務効率化に向けての課題発見力＞\n\n業務効率化への提案および実施に向けてのフロー・マニュアルの制作に数多く携わりました。業務課題は、トラブル時などのイレギュラー対応が発生した際に顕在化することが多いです。\n\n対応完了まで迅速に取り組むだけでなく、「なぜそのトラブルが発生したのか」についての振り返りと再発防止策の検討に率先して取り組んでいました。\n\nまた、組織が変わっても実践できるよう、トラブル対応策をマニュアルにまとめてきました。\n\n前職では、売掛金・買掛金管理において、見落としなどの属人的なミスを防止するためのチェックフロー体制を整えた経験があります。\n\n結果として、15％の時間削減とミスのない経理業務を実現できました。', 'confidence': 1}}


#### Measure time and cost it take to process document in excel format

In [83]:
path_to_sample_document = "./examples/xlsx/職務経歴書(サンプル).xlsx"

In [88]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2 page xlsx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

13.6 s ± 2.18 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [84]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [85]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 2, 'contextualizationTokens': 2000, 'tokens': {'text-embedding-3-large': 5264, 'gpt-5-input': 9359, 'gpt-5-output': 854}}

In [86]:
# Calulate total cost in usd for running cv parsing per document (2-page xlsx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.023090462


In [87]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array'},
 'スキル': {'type': 'object', 'valueObject': {'技術': {'type': 'string'}, '言語': {'type': 'string'}}},
 '自己PR': {'type': 'string', 'valueString': '＜業務における迅速性と正確性＞迅速性と正確性を心がけています。 電子データ・紙媒体それぞれの情報を必要な時にすぐに取り出せるように適切に整理し、また効率的に作業を進められるよう Excel を活用してデータ管理をしていました。また就業開始時には必ずスケジュール進捗とタスクの優先度整理 を行い、業務に臨みました。', 'confidence': 1}}


### Under api_version=2026-06-01-preview with gpt-5-mini

In [9]:
# API_VERSION - the API version to use.
api_version = analyer_candidates_list[2]["api_version"]

In [10]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = analyer_candidates_list[2]["analyzer_id"]

In [11]:
# Set up Content Understanding client.
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

#### Measure time and cost it take to process document in pdf format

In [15]:
# local file localtion
path_to_sample_document = "./examples/pdf/職務経歴書(サンプル).pdf"

In [20]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page pdf)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

20.8 s ± 4.34 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [17]:
# usage of tokens
poller.usage

{'documentPagesStandard': 2, 'advancedContextualizationTokens': 2000, 'tokens': {'gpt-5-mini-input': 8878, 'gpt-5-mini-output': 1565}}

In [18]:
# Calulate total cost in usd for running cv parsing per document (2-page pdf)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.0213495


In [19]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社▲▲', 'spans': [{'offset': 72, 'length': 6}], 'confidence': 0.334, 'source': 'D(1,1.3828,2.1819,2.1591,2.1816,2.1592,2.3340,1.3829,2.3343)'}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'confidence': 0.953}, '終了日': {'type': 'date', 'confidence': 0.954}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'spans': [{'offset': 293, 'length': 11}], 'confidence': 0.409, 'source': 'D(1,1.5981,4.4793,3.0315,4.4889,3.0304,4.6523,1.5970,4.6427)'}, '業種': {'type': 'string', 'confidence': 0.956}, '職種': {'type': 'string', 'confidence': 0.957}, '雇用形態': {'type': 'string', 'valueString': '正社員', 'spans': [{'offset': 283, 'length': 3}], 'confidence': 0.496, 'source': 'D(1,4.4338,3.9979,4.8381,3.9969,4.8381,4.1459,4.4338,4.1466)'}, '職務内容': {'type': 'string', 'confidence': 0.957}, '実績と成果': {'type': 'string', 'confidence': 0.956}, '

#### Measure time and cost it take to process document in docx format

In [21]:
path_to_sample_document = "./examples/docx/職務経歴書(サンプル).docx"

In [26]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2 page docx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

13.7 s ± 1.34 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [23]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 1, 'advancedContextualizationTokens': 1000, 'tokens': {'gpt-5-mini-input': 7192, 'gpt-5-mini-output': 1213}}

In [24]:
# Calulate total cost in usd for running cv parsing per document (2-page docx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.007233999999999999


In [25]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社▲▲', 'confidence': 1}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'confidence': 1}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'confidence': 1}}}, '事業内容': {'type': 'string', 'valueString': '事業内容：インターネット広告事業 資本金：3 千万円（2015 年度） 売上高：1 億 9 千万円（2015 年 3 月）従業員数：100 人 上場区分：未上場', 'confidence': 1}, '業種': {'type': 'string'}, '職種': {'type': 'string', 'valueString': '主任（総務経理課）', 'confidence': 1}, '雇用形態': {'type': 'string', 'valueString': '正社員', 'confidence': 1}, '職務内容': {'type': 'string', 'valueString': '仕訳、伝票処理（売掛金、買掛金）、現金出納管理、預金口座管理、年末調整、償却資産管理、月次・年次財務諸表作成（貸借対照表、損益計算書、キャッシュフロー計算書）、原価計算配賦計算（原材料/人件費/経費）、月次部門別損益集計', 'confidence': 1}, '実績と成果': {'type': 'string'}, '所属部門': {'type': 'string', 'valueString': '東京本社 管理部 経理課 → 大阪支店 管理部 総務経理課', 'confidence': 1}}}, {'type': 'object', 'valueObject': {'会社名':

#### Measure time and cost it take to process document in excel format

In [27]:
path_to_sample_document = "./examples/xlsx/職務経歴書(サンプル).xlsx"

In [28]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2 page xlsx)
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

14.1 s ± 1.49 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
# implement cv parser on given cv
result, poller = parse_cv(analyzer_id, path_to_sample_document, client)

In [30]:
# usage of tokens
poller.usage

{'documentPagesMinimal': 2, 'advancedContextualizationTokens': 2000, 'tokens': {'gpt-5-mini-input': 9320, 'gpt-5-mini-output': 1010}}

In [31]:
# Calulate total cost in usd for running cv parsing per document (2-page xlsx)
caluate_total_cost(poller, price_dict)

Total cost (usd) per document(2-page pdf): 0.01037


In [32]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社▲▲', 'confidence': 1}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'confidence': 1}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'confidence': 1}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業 資本金：3 千万円（2015 年度） 売上高：1 億 9 千万円（2015 年 3 月）従業員数：100 人 上場区分：未上場', 'confidence': 1}, '業種': {'type': 'string'}, '職種': {'type': 'string'}, '雇用形態': {'type': 'string', 'valueString': '正社員', 'confidence': 1}, '職務内容': {'type': 'string', 'valueString': '仕訳、伝票処理（売掛金、買掛金）・現金出納管理、預金口座管理・年末調整、償却資産管理・月次・年次財務諸表作成（貸借対照表、損益計算書、キャッシュフロー計算書）・原価計算配賦計算（原材料/人件費/経費）・月次部門別損益集計', 'confidence': 1}, '実績と成果': {'type': 'string', 'valueString': '※記載なし', 'confidence': 1}, '所属部門': {'type': 'string', 'valueString': '管理部 経理課（東京本社）→管理部 総務経理課（大阪支店）', 'confidence': 1}}}, {'type': 'object', 'valueObject': {'会社名': {'type':